# reFuel to MOTEL unmapped records

This notebook structures the conversion of reFuel technology rows into MOTEL-compatible unmapped records.

## Goals
- Read reFuel input data from Excel.
- Load and inspect the MOTEL schema.
- Map each source row into a normalized record payload.
- Validate required fields and export records for review.

## Output target
- Write records to `dev2/motel-db/records/` as JSON or YAML files.

In [77]:
save_to_db = True

In [78]:
## --- load packages
from pathlib import Path
import pandas as pd
import yaml

# 1) Inputs and setup

## 1.1 Source data (reFuel Excel)

In [79]:
path_refuel = Path(r"E:\Barton\repositories\motel-platform\dev2\data\reFuel_TechDatabase_2026-06-03.xlsx")

refuel_data = pd.read_excel(path_refuel, sheet_name=None)
print(f"sheets found: {list(refuel_data.keys())}")

sheets found: ['README', 'Metadata', 'Predefined', 'Unique values', 'Glossary', 'Carrier', 'Location', 'Currency', 'Process', 'Process ID', 'Technology ID', 'LCA ID', 'Reference ID', 'SolarTech', 'ConvTech', 'StorTech', 'TechMaturity', 'ConvTech(old)', 'Uncertainty', 'Tech Constraints', 'ImportExport Constraints', 'Resource Constraints', 'CO2 Constraints', 'Tech data (long)', 'System setup', 'MOTEL']


## 1.2 Load MOTEL schema

In [80]:
path_schema = r'schema\motel_schema.yaml'

with open(path_schema, "r", encoding="utf-8") as f:
    schema_motel = yaml.safe_load(f)

print(f"Schema loaded successfully")
print(f"Type: {type(schema_motel)}")
if isinstance(schema_motel, dict):
    print(f"Top-level keys ({len(schema_motel)}): {list(schema_motel.keys())[:10]}")

Schema loaded successfully
Type: <class 'dict'>
Top-level keys (13): ['unmapped_record', 'mapped_record', 'technology', 'process', 'source', 'contributor', 'review', 'attribute', 'carrier', 'geographic_scope']


# 2) Create secondary data from Refuel.ch data

## 2.1 Technology and process datasets (done)

source: sheet 'Technology ID'

In [81]:
sheet_name = 'Technology ID'
df_tech_id = refuel_data[sheet_name].copy()

## correct column names
df_tech_id.columns = df_tech_id.loc[1]
df_tech_id = df_tech_id.loc[2:].reset_index(drop=True)

df_tech_id

1,sheet,tech_id,ehubX Tech ID,process_id,process_type,unit_operation,tech_ontology_IRI,description,tech_year,main_sector,main_category,category_spec,tech_type
0,ConvTech,Ammonia_HaberBosch,T_C_Conv_Ammonia_HaberBosch,Ammonia_HaberBosch,Ammonia Synthesis,Haber Bosch unit,,Ammonia synthesis via Haber Bosch process.,2050,Ammonia,Chemical Synthesis,Haber Bosch,Conversion
1,ConvTech,Ammonia_OCGT,T_C_Conv_Ammonia_OCGT,Ammonia_OCGT,Electricity Generation,OCGT,,,2050,Electricity,Gas Turbine,Ammonia fueled,Conversion
2,ConvTech,Amine_Washing_CO2,T_C_Conv_CarbonCapture_Amine,CarbonCapture_Amine,CO2 Capture,Amine washing unit,,CO2 post combustion capture via Amine washing....,"2030, 2040, 2050",Carbon Capture,Point Source Capture,Amine washing,Conversion
3,ConvTech,Waste_Venting_CO2,T_C_Conv_CarbonCapture_CO2Waste_Vent,CarbonCapture_CO2Waste_Vent,CO2 Capture,Ventilator,,CO2 venting from waste-related processes.,2050,Carbon Capture,Ventilation,Waste,Conversion
4,ConvTech,Wood_Venting_CO2,T_C_Conv_CarbonCapture_CO2Wood_Vent,CarbonCapture_CO2Wood_Vent,CO2 Capture,Ventilator,,CO2 venting from wood-related processes.,2050,Carbon Capture,Ventilation,Wood,Conversion
...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,StorTech,CarbonDioxide_Tank,T_S_CarbonDioxide_Tank,NaN,Storage,Tank,,NaN,na,Carbo Dioxide,,,Storage
91,StorTech,CarbonMonoxide_Tank,T_S_CarbonMonoxide_Tank,NaN,Storage,Tank,,NaN,na,Carbon Monoxide,,,Storage
92,StorTech,C8_Hydrocarbons_Tank,T_S_C8_Hydrocarbons_Tank,NaN,Storage,Tank,,NaN,na,Hydrocarbons,,,Storage
93,StorTech,C12_Hydrocarbons_Tank,T_S_C12_Hydrocarbons_Tank,NaN,Storage,Tank,,NaN,na,Hydrocarbons,,,Storage


In [82]:
df_proc_id = refuel_data['Process ID'].copy()
df_proc_id.columns = df_proc_id.loc[0]
df_proc_id = df_proc_id.loc[1:].reset_index(drop=True)

df_proc_id

,Process ID,Process Description,Tech Type,Process Category,Primay Feedstock,Feedstocks,Products,Reference,Assumption
0,Ammonia_HaberBosch,Synthesizing Ammonia from Haber Bosch Process,Conversion,Haber Bosch,NaN,NaN,NaN,NaN,NaN
1,biomass_gasification_to_syngas,Biomass gasification to syngas,Conversion,Gasification,Syngas,"Wood, O2",H2,NaN,NaN
2,CarbonCapture_Amine,CO2 post combustion capture via Amine washing....,Conversion,Carbon Capture,NaN,"CO2, amine",NaN,NaN,NaN
3,CarbonCapture_CO2Waste_Vent,CO₂ venting from waste-related processes.,Conversion,Carbon Capture,NaN,waste,NaN,NaN,NaN
4,CarbonCapture_CO2Wood_Vent,CO₂ venting from wood-related processes.,Conversion,Carbon Capture,NaN,wood,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
80,ThHt_Vent,Ventilation heat loss at high-temperature heat...,Conversion,Heat loss,NaN,heat,NaN,NaN,NaN
81,ThLt_Vent,Ventilation heat loss at low-temperature heat ...,Conversion,Heat loss,NaN,heat,NaN,NaN,NaN
82,ThPh_Vent,Ventilation heat loss at thermal power plants.,Conversion,Heat loss,NaN,heat,NaN,NaN,NaN
83,wet_biomass_anaerobic_digestion_gas_separation,Wet biomass anaerobic digestion with gas separ...,Conversion,Digestion,Methane,BioWet,"CH4, CO2",NaN,NaN


In [83]:
# Technology Attributes
map_tech = {    
    'tech_name': ('Technology ID', 'tech_id'),
    'main_operation_unit': ('Technology ID', 'unit_operation'),
    'technology_description': ('Technology ID', 'description'),
    'technology_variant': ('Technology ID', 'category_spec'),
}

# Process Attributes
map_process = {    
    'process_name': ('Process ID', 'Process ID'),
    'process_description': ('Process ID', 'Process Description'),
    'process_type': ('Process ID', 'Tech Type'),          # Note: Old tech_type -> New process_type
    'process_category': ('Process ID', 'Process Category'),
    'feedstocks': ('Process ID', 'Feedstocks'),
    'products': ('Process ID', 'Products'),
    
    'main_sector': ('Technology ID', 'main_sector')
}

In [84]:
# Build mapped Technology and Process dataframes from existing mapping dictionaries

source_frames = {
    "Technology ID": df_tech_id,
    "Process ID": df_proc_id,
}

def build_mapped_df(mapping, source_frames):
    out = pd.DataFrame()
    for target_col, (sheet_name, source_col) in mapping.items():
        df_src = source_frames[sheet_name]
        out[target_col] = df_src[source_col] if source_col in df_src.columns else pd.NA
    return out

# 1) Technology dataframe
df_technology = build_mapped_df(map_tech, source_frames)

# 2) Process dataframe (base fields from Process ID + main_sector from Technology ID)
df_process = build_mapped_df(
    {k: v for k, v in map_process.items() if v[0] == "Process ID"},
    source_frames
)

# Derive process -> main_sector lookup from Technology ID sheet
process_sector_lookup = (
    df_tech_id[["process_id", "main_sector"]]
    .dropna(subset=["process_id"])
    .drop_duplicates(subset=["process_id"])
    .set_index("process_id")["main_sector"]
)

df_process["main_sector"] = df_process["process_name"].map(process_sector_lookup)

# Optional cleanup
df_process = df_process.drop_duplicates().reset_index(drop=True)
df_technology = df_technology.drop_duplicates().reset_index(drop=True)

# add id at the first column with format (tech_00001)
df_technology.insert(0, "tech_id", df_technology.index.map(lambda x: f"tech_{x+1:05d}"))
df_process.insert(0, "process_id", df_process.index.map(lambda x: f"proc_{x+1:05d}"))

print("Technology dataframe shape:", df_technology.shape)
print("Process dataframe shape:", df_process.shape)

display(df_technology.head())
display(df_process.head())

Technology dataframe shape: (95, 5)
Process dataframe shape: (85, 8)


,tech_id,tech_name,main_operation_unit,technology_description,technology_variant
0,tech_00001,Ammonia_HaberBosch,Haber Bosch unit,Ammonia synthesis via Haber Bosch process.,Haber Bosch
1,tech_00002,Ammonia_OCGT,OCGT,,Ammonia fueled
2,tech_00003,Amine_Washing_CO2,Amine washing unit,CO2 post combustion capture via Amine washing....,Amine washing
3,tech_00004,Waste_Venting_CO2,Ventilator,CO2 venting from waste-related processes.,Waste
4,tech_00005,Wood_Venting_CO2,Ventilator,CO2 venting from wood-related processes.,Wood


,process_id,process_name,process_description,process_type,process_category,feedstocks,products,main_sector
0,proc_00001,Ammonia_HaberBosch,Synthesizing Ammonia from Haber Bosch Process,Conversion,Haber Bosch,NaN,NaN,Ammonia
1,proc_00002,biomass_gasification_to_syngas,Biomass gasification to syngas,Conversion,Gasification,"Wood, O2",H2,NaN
2,proc_00003,CarbonCapture_Amine,CO2 post combustion capture via Amine washing....,Conversion,Carbon Capture,"CO2, amine",NaN,Carbon Capture
3,proc_00004,CarbonCapture_CO2Waste_Vent,CO₂ venting from waste-related processes.,Conversion,Carbon Capture,waste,NaN,Carbon Capture
4,proc_00005,CarbonCapture_CO2Wood_Vent,CO₂ venting from wood-related processes.,Conversion,Carbon Capture,wood,NaN,Carbon Capture


In [85]:
# export to csv

if save_to_db:
    df_technology.to_csv(r"motel-db\secondary\technology.csv", index=False)
    print(f'Technology data saved to CSV')

    df_process.to_csv(r"motel-db\secondary\process.csv", index=False)
    print(f'Process data saved to CSV')


Technology data saved to CSV
Process data saved to CSV


## 2.2 Source (done)

In [86]:
# load source data from reFuel.ch data
df_ref = refuel_data["Reference ID"].copy()
df_ref = df_ref.loc[1:].reset_index(drop=True)

# convert to motel-db source format
access_date_iso = pd.to_datetime(df_ref["Access Date"], errors="coerce").dt.strftime("%Y-%m-%d")
access_date_iso = access_date_iso.where(pd.notna(access_date_iso), None)

records_source = [
    {
        "source_name": row["Reference ID"],
        "source_description": row["Reference Description"] if pd.notna(row["Reference Description"]) else None,
        "source_type": None,
        "link": (
            None
            if pd.isna(row["DOI or Link"]) or str(row["DOI or Link"]).strip().lower() in {"na", "nan", ""}
            else str(row["DOI or Link"]).strip()
        ),
        "access_date": access_date_iso.iloc[i],
        "confidence_level": None,
        "assessment_method": None,
        "assessment_date": None,
    }
    for i, row in df_ref.iterrows()
]

df_source = pd.DataFrame(records_source)

df_source.insert(0, "source_id", df_source.index.map(lambda x: f"src_{x+1:05d}"))

df_source

,source_id,source_name,source_description,source_type,link,access_date,confidence_level,assessment_method,assessment_date
0,src_00001,report_VSE_2022,"VSE Energiezukunft 2050, Studie 2022",None,https://www.strom.ch/de/energiezukunft-2050/do...,2023-09-06,None,None,None
1,src_00002,assumption_Empa_2022,Expert Assumption,None,NaN,NaN,None,None,None
2,src_00003,paper_mutschler_2018,NaN,None,https://doi.org/10.1016/j.jcat.2018.08.002,NaN,None,None,None
3,src_00004,assumption_reFuelWP3_2025,Expert Assumption,None,NaN,NaN,None,None,None
4,src_00005,report_VSE_2025,"VSE Energiezukunft 2050, Update 2025",None,https://www.strom.ch/de/energiezukunft-2050/do...,2025-05-26,None,None,None
5,src_00006,paper_müller_PowerX_2025,The costs of future energy technologies: A com...,None,https://doi.org/10.1016/j.jcou.2025.103019,NaN,None,None,None
6,src_00007,assumption_Empa_2025,Expert assumptions and calculations by Empa,None,NaN,NaN,None,None,None
7,src_00008,paper_alpinePV_2024,ETHZ paper on Alpine PV cost and financial via...,None,https://doi.org/10.1016/j.apenergy.2024.124019,NaN,None,None,None
8,src_00009,paper_PVcostProjection_2025,"review the cost projection of PV, wind, battery",None,https://doi.org/10.1016/j.apenergy.2025.125856,NaN,None,None,None
9,src_00010,assumption_PVcostProjection,4/3/2% cost reduction per year in 2030/2040/20...,None,NaN,NaN,None,None,None


In [87]:
# export to csv

if save_to_db:
    df_source.to_csv(r"motel-db\secondary\source.csv", index=False)
    print(f'Source data saved to CSV')

Source data saved to CSV


## 2.3 Attribute (to be checked)

exclude attirbutes for balancing, deal with it for balancing later

In [88]:
## load the pre-defined mapping for attribute

path_attr_mapping = r"data\refuel2motel_map\attribute.csv"

df_attr_mapping = pd.read_csv(path_attr_mapping)
df_attr_mapping

,Raw Attribute,attribute_name,attribute_description,unit,data_format,attribute_note
0,overall_efficiency,efficiency_overall,Ratio of total useful output energy to total i...,-,Float,Standard efficiency metric.
1,theoretical_efficiency,efficiency_theoretical,Maximum thermodynamic efficiency (Carnot limit).,-,Float,Benchmark for performance.
2,operating_temperature_c,temp_operating,Typical operating temperature of the process.,°C,Float,Critical Fix: Was mapped to discount rate in s...
3,min_installation_size,capacity_min,Minimum viable installation capacity.,kW or kg/h,Float,Critical Fix: Was mapped to Value Range Check.
4,lifetime_yr,lifetime_economic,Economic lifetime of the asset.,yr,Integer,Critical Fix: Was mapped to uncertainty rating.
5,capex_one_time_eur,capex_fixed,Fixed capital expenditure (independent of size).,CUR,Float,Critical Fix: Was mapped to LCA ID.
6,capex_power_capacity_eur_per_kw,capex_variable,Variable capital expenditure per unit of capac...,CUR/kW,Float,Critical Fix: Was mapped to LCA unit.
7,opex_one_time_eur,opex_fixed,Fixed operating expenditure (annual).,CUR/yr,Float,Critical Fix: Was mapped to Embedded Carbon.
8,opex_fix_pct_of_capex,opex_fixed_pct,Fixed OPEX expressed as % of Total CAPEX.,%,Float,Alternative definition for opex_fixed.
9,opex_fix_power_capacity_eur_per_kw_yr,opex_variable,Variable OPEX per unit of capacity per year.,CUR/kW/yr,Float,Critical Fix: Was mapped to References.


# 3) Create record from reFuel.ch data sheets

TODO: adept the conversion for balancing

## 3.1 Function Layers for Row -> Record Conversion

This notebook intentionally uses two function layers:

- Layer 1 (next code cell): base conversion logic from one reFuel row to one record, including final balancing schema construction.
- Layer 2 (following code cell): post-processing overrides that re-define selected functions (for example `add_value` and `map_technology`) to add filtering, unit parsing, and currency replacement.

Run the two code cells in order so Layer 2 extends Layer 1 correctly.

In [89]:
# LAYER 1 (BASE): core row -> record mapping.
# This cell defines the base implementation of map_technology and helpers.
# Balancing is built directly in final target schema here.
## --- function to convert reFuel.ch data into 'unmapped record'

import math
import re


def is_nan(value):
    return value is None or (
        isinstance(value, float) and math.isnan(value)
    )


def clean(value):
    return None if is_nan(value) else value


def parse_number_with_unit(text):
    """
    Converts:
        '2800 CHF/kg/h'
    into:
        (2800.0, 'CHF/kg/h')
    """
    if not isinstance(text, str):
        return text, None

    match = re.match(r"^\s*([0-9.]+)\s*(.*)$", text)
    if match:
        return float(match.group(1)), match.group(2).strip()

    return text, None


def split_csv(value):
    if is_nan(value):
        return []
    return [
        x.strip()
        for x in str(value).split(",")
        if x.strip() and x.strip().lower() not in {"na", "nan"}
    ]


def split_csv_float(value):
    out = []
    for token in split_csv(value):
        try:
            out.append(float(token))
        except ValueError:
            out.append(None)
    return out


def split_number_unit(value):
    """Split strings like '20000 kg/h' into (20000.0, 'kg/h').

    Returns (original_value, None) when no numeric+unit pattern is found.
    """
    if not isinstance(value, str):
        return value, None

    text = value.strip()
    if not text:
        return value, None

    match = re.match(r"^([-+]?\d[\d,]*(?:\.\d+)?)\s+(.+)$", text)
    if not match:
        return value, None

    number_text, unit_text = match.groups()
    unit_text = unit_text.strip()
    if not unit_text:
        return value, None

    try:
        numeric_value = float(number_text.replace(",", ""))
    except ValueError:
        return value, None

    return numeric_value, unit_text


def is_empty_like_base(value):
    """Return True for empty / NA-like values used for final record pruning."""
    if is_nan(value):
        return True

    if isinstance(value, str):
        token = value.strip().lower()
        return token in {"", "na", "nan", "none", "null", "n/a"}

    if isinstance(value, (list, tuple, set)):
        return len(value) == 0 or all(is_empty_like_base(v) for v in value)

    if isinstance(value, dict):
        return len(value) == 0 or all(is_empty_like_base(v) for v in value.values())

    return False


def is_empty_like(value):
    """Return True for empty / NA-like values.

    Empty-like values include None, NaN, empty strings, and textual NA markers.
    Containers are considered empty when all contained values are empty-like.
    """
    if is_nan(value):
        return True

    if isinstance(value, str):
        token = value.strip().lower()
        return token in {"", "na", "nan", "none", "null", "n/a"}

    if isinstance(value, (list, tuple, set)):
        return len(value) == 0 or all(is_empty_like(v) for v in value)

    if isinstance(value, dict):
        return len(value) == 0 or all(is_empty_like(v) for v in value.values())

    return False


def add_value(values, name, description, value, unit=None, scenario=None, time_index=None):
    """Append one attribute if and only if value is non-empty.

    Also supports overriding default unit when value embeds unit text,
    e.g., value='20000 kg/h' with default unit='kW or kg/h'.
    """
    if is_empty_like(value):
        return

    parsed_value, parsed_unit = split_number_unit(value)
    if parsed_unit:
        value = parsed_value
        unit = parsed_unit

    record = {
        "attribute_name": name,
        "attribute_description": description,
        "unit": unit,
        "value": value,
    }

    if scenario:
        record["scenario"] = scenario

    if time_index:
        record["time_index"] = time_index

    values.append(record)


def build_balancing_entries(carriers, shares, units):
    size = max(len(carriers), len(shares), len(units))
    entries = []

    for i in range(size):
        carrier = carriers[i] if i < len(carriers) else None
        share = shares[i] if i < len(shares) else None
        unit = units[i] if i < len(units) else None

        if carrier is None and share is None and unit is None:
            continue

        entries.append({
            "carrier": carrier,
            "share": share,
            "unit": unit,
        })

    return entries


def to_balance_list(items):
    """Normalize balancing items to list of dicts with carrier_name/share/unit."""
    normalized = []
    for item in items or []:
        if not isinstance(item, dict):
            continue

        carrier_name = clean(item.get("carrier_name"))
        if carrier_name is None:
            carrier_name = clean(item.get("carrier"))

        share = clean(item.get("share"))
        unit = clean(item.get("unit"))

        if carrier_name is None and share is None and unit is None:
            continue

        normalized.append({
            "carrier_name": carrier_name,
            "share": share,
            "unit": unit,
        })

    return normalized


def prune_empty_keys(obj):
    """Recursively remove keys/items that are empty-like."""
    if isinstance(obj, dict):
        cleaned = {}
        for key, value in obj.items():
            pruned = prune_empty_keys(value)
            if not is_empty_like_base(pruned):
                cleaned[key] = pruned
        return cleaned

    if isinstance(obj, list):
        cleaned_list = []
        for value in obj:
            pruned = prune_empty_keys(value)
            if not is_empty_like_base(pruned):
                cleaned_list.append(pruned)
        return cleaned_list

    return obj


def normalize_unit(unit_text):
    if is_nan(unit_text):
        return None
    unit = str(unit_text).strip()
    if not unit or unit == "-":
        return None
    return unit


def split_or_units(unit_text):
    """Split expressions like 'kW or kg/h' into candidate units."""
    if not isinstance(unit_text, str):
        return []
    return [u.strip() for u in re.split(r"\s+or\s+", unit_text, flags=re.IGNORECASE) if u.strip()]


def infer_main_output_unit(main_output_carrier, output_carriers, output_units):
    """Infer the unit for the selected main output carrier."""
    if main_output_carrier is not None:
        for carrier, unit in zip(output_carriers, output_units):
            if clean(carrier) == main_output_carrier:
                unit_norm = normalize_unit(unit)
                if unit_norm:
                    return unit_norm

    # Fallback: first non-empty output unit if main output is not matched.
    for unit in output_units:
        unit_norm = normalize_unit(unit)
        if unit_norm:
            return unit_norm

    return None


def choose_preferred_or_unit(unit_text, main_output_unit):
    """Pick one unit from a disjunctive unit string using output-unit context."""
    candidates = split_or_units(unit_text)
    if not candidates:
        return normalize_unit(unit_text)

    if not main_output_unit:
        return candidates[0]

    target = str(main_output_unit).strip().lower()
    candidates_lower = [c.lower() for c in candidates]

    # Exact match first.
    for idx, cand_lower in enumerate(candidates_lower):
        if cand_lower == target:
            return candidates[idx]

    # Heuristics for common power vs mass-flow disambiguation.
    if ("kg/" in target) or target.startswith("kg") or ("ton/" in target) or target.startswith("t/"):
        for idx, cand_lower in enumerate(candidates_lower):
            if "kg/" in cand_lower or cand_lower.startswith("kg") or "ton/" in cand_lower or cand_lower.startswith("t/"):
                return candidates[idx]

    if ("w" in target) or ("wh" in target) or ("j/s" in target):
        for idx, cand_lower in enumerate(candidates_lower):
            if ("w" in cand_lower) or ("wh" in cand_lower) or ("j/s" in cand_lower):
                return candidates[idx]

    return candidates[0]


def resolve_attribute_unit(attribute_name, unit_text, main_output_unit):
    """Resolve attribute unit; supports context-based choice for 'A or B' units."""
    unit_norm = normalize_unit(unit_text)
    if not unit_norm:
        return None

    # Keep this generic so future attributes with disjunctive units also work.
    if isinstance(unit_norm, str) and re.search(r"\s+or\s+", unit_norm, flags=re.IGNORECASE):
        return choose_preferred_or_unit(unit_norm, main_output_unit)

    return unit_norm


def log_main_output_unit_resolution(tech_name, main_output_carrier, main_output_unit, specs):
    """Print a trace line showing inferred main-output unit and resolved capacity default."""
    capacity_unit_options = None
    for spec in specs:
        if spec.get("attribute_name") == "capacity_min":
            raw_unit = spec.get("unit")
            if isinstance(raw_unit, str) and re.search(r"\s+or\s+", raw_unit, flags=re.IGNORECASE):
                capacity_unit_options = raw_unit
                break

    if capacity_unit_options:
        selected_capacity_unit = choose_preferred_or_unit(capacity_unit_options, main_output_unit)
    else:
        selected_capacity_unit = None

    # print(
    #     f"[unit-resolution] tech '{tech_name}' has main_output '{main_output_carrier}' "
    #     f"with unit '{main_output_unit}', and default capacity unit '{selected_capacity_unit}' is used."
    # )


def build_attribute_specs(mapping_df):
    required_cols = {"Raw Attribute", "attribute_name", "attribute_description", "unit"}
    if not isinstance(mapping_df, pd.DataFrame) or not required_cols.issubset(mapping_df.columns):
        return []

    specs = []
    for _, rec in mapping_df.iterrows():
        raw_attr = clean(rec.get("Raw Attribute"))
        attr_name = clean(rec.get("attribute_name"))

        if not raw_attr or not attr_name:
            continue

        specs.append({
            "raw_attribute": str(raw_attr),
            "attribute_name": str(attr_name),
            "attribute_description": clean(rec.get("attribute_description")),
            "unit": normalize_unit(rec.get("unit")),
        })

    # De-duplicate by target attribute name, keep first mapping row.
    deduped = {}
    for spec in specs:
        deduped.setdefault(spec["attribute_name"], spec)

    return list(deduped.values())


def get_mapped_raw_value(row, raw_attribute, input_carriers, output_carriers):
    # This row in mapping is conceptual; derive from parsed carriers.
    if raw_attribute == "Input, Output for Overall Efficiency":
        combined = input_carriers + output_carriers
        return combined if combined else None

    return clean(row.get(raw_attribute))


def map_technology(row):

    result = {
        "technology_name": (
            row.get("tech_id", "")
            .replace("_", " ")
        ),        

        "technology": {
            "technology_description": clean(row.get("description")),
            "technology_type": clean(row.get("tech_type")),
            "technology_category": clean(row.get("main_category")),
            "technology_assumption": (
                f"{row.get('cost_base_year')} reference technology"
                if not is_nan(row.get("cost_base_year"))
                else None
            ),
            "process_name": clean(row.get("unit_operation")),
            "process_type": clean(row.get("process_type")),
            "process_category": clean(row.get("main_sector")),
            "process_assumption": clean(row.get("category_spec"))
        },

        "scope": {
            "geographic_scope_description":
                clean(row.get("Cost Base")),
            "temporal_scope_description":
                str(row.get("cost_base_year"))
                if not is_nan(row.get("cost_base_year"))
                else None,
            "capacity_scope_description":
                clean(row.get("min_installation_size")),
            "system_boundary_description":
                clean(row.get("tech_boundary")),
            "scope_assumption":
                clean(row.get("tech_maturity"))
        },

        "sources": [],
        "attributes": [],
        "balancing": {
            "inputs": [],
            "main_input": clean(row.get("main_input")),
            "outputs": [],
            "main_output": clean(row.get("main_out")),
            "balance_assumption": clean(row.get("balance_assumption")),
        },
        "tags": {
            "related_project": "reFuel.ch",
            "tags": []
        }
    }

    attributes = result["attributes"]

    # -----------------------
    # Source parsing
    # -----------------------
    source_text = row.get("list_of_source_id")

    if isinstance(source_text, str):

        if "AECOM_2022" in source_text:
            result["sources"].append({
                "source_description": "AECOM_2022",
                "source_type": "report",
                "assessment_method": "literature"
            })

        if "parameters_lca" in source_text:
            result["sources"].append({
                "source_description": "parameters_lca",
                "source_type": "database",
                "assessment_method": "modeled"
            })

    # -----------------------
    # Inputs and outputs for balancing
    # -----------------------
    input_carriers = split_csv(row.get("carriers_in"))
    input_shares = split_csv_float(row.get("ratios_in"))
    input_units = split_csv(row.get("units_in"))

    output_carriers = split_csv(row.get("carriers_out"))
    output_shares = split_csv_float(row.get("ratios_out"))
    output_units = split_csv(row.get("units_out"))

    result["balancing"]["inputs"] = to_balance_list(
        build_balancing_entries(
            input_carriers,
            input_shares,
            input_units,
        )
    )
    result["balancing"]["outputs"] = to_balance_list(
        build_balancing_entries(
            output_carriers,
            output_shares,
            output_units,
        )
    )

    main_output_carrier = clean(row.get("main_out"))
    main_output_unit = infer_main_output_unit(
        main_output_carrier,
        output_carriers,
        output_units,
    )

    # -----------------------
    # Attributes from mapping CSV only
    # -----------------------
    specs = build_attribute_specs(df_attr_mapping)
    log_main_output_unit_resolution(
        result.get("technology_name"),
        main_output_carrier,
        main_output_unit,
        specs,
    )
    for spec in specs:
        value = get_mapped_raw_value(
            row,
            spec["raw_attribute"],
            input_carriers,
            output_carriers,
        )

        resolved_unit = resolve_attribute_unit(
            spec["attribute_name"],
            spec["unit"],
            main_output_unit,
        )

        add_value(
            attributes,
            spec["attribute_name"],
            spec["attribute_description"],
            value,
            unit=resolved_unit,
        )

    # -----------------------
    # Tags
    # -----------------------
    tags = result["tags"]["tags"]

    for field in [
        "main_sector",
        "main_category",
        "category_spec",
        "tech_type",
        "tech_maturity"
    ]:
        value = clean(row.get(field))
        if value:
            tags.append(str(value))

    return prune_empty_keys(result)

In [90]:
# LAYER 2 (OVERRIDE): intentional post-processing overrides.
# This cell redefines selected helpers and wraps map_technology from Layer 1.
# Purpose: empty filtering, embedded unit parsing, and CUR replacement in units.

def apply_currency_to_units(record, row):
    """Replace CUR placeholder in attribute units with row-level currency."""
    currency = clean(row.get("Currency"))
    if not isinstance(currency, str):
        return record

    currency = currency.strip()
    if not currency:
        return record

    for item in record.get("values", []):
        unit = item.get("unit")
        if isinstance(unit, str) and "CUR" in unit:
            item["unit"] = unit.replace("CUR", currency)

    return record


# Keep wrapper idempotent across re-runs by retaining a stable base function.
_current_map_technology = map_technology
if getattr(_current_map_technology, "_is_wrapper", False):
    _base_map_technology = _current_map_technology._base_map_technology
else:
    _base_map_technology = _current_map_technology


def map_technology(row):
    """Wrapper: build record and apply unit post-processing."""
    rec = _base_map_technology(row)
    rec = apply_currency_to_units(rec, row)
    return rec


map_technology._is_wrapper = True
map_technology._base_map_technology = _base_map_technology

### Check the content of record

In [91]:
# run record crating for a testing row

## load the data from ConvTech sheet for testing
df_conv = refuel_data['ConvTech'].copy()
df_conv.columns = df_conv.loc[2]
df_conv = df_conv.loc[3:].reset_index(drop=True)

row = df_conv.loc[10]

## run record mapping for the testing row
record = map_technology(row)

In [92]:
# quick check: new balancing payload
record.get("balancing")

{'inputs': [{'carrier_name': 'CO2Fluegas', 'share': 1.0, 'unit': 'kg'},
  {'carrier_name': 'El13', 'share': 0.52, 'unit': 'kWh'},
  {'carrier_name': 'ThHt', 'share': 0.024, 'unit': 'kWh'}],
 'main_input': 'CO2Fluegas',
 'outputs': [{'carrier_name': 'CO2Sys', 'share': 0.95, 'unit': 'kg'}],
 'main_output': 'CO2Sys'}

In [93]:
# quick check: attributes come only from mapping CSV
mapped_attr_names = set(df_attr_mapping["attribute_name"].dropna().astype(str).str.strip())
record_attr_names = {v["attribute_name"] for v in record.get("values", [])}

print("record attributes:", len(record_attr_names))
print("not in mapping:", sorted(record_attr_names - mapped_attr_names))

record attributes: 0
not in mapping: []


In [94]:
# quick check: no CUR placeholder should remain after currency replacement
cur_units = [
    v.get("unit")
    for v in record.get("values", [])
    if isinstance(v.get("unit"), str) and "CUR" in v.get("unit")
]

print("remaining CUR units:", cur_units)
print("row currency:", row.get("Currency"))

remaining CUR units: []
row currency: CHF


In [95]:
record.get('values')

## 3.2 Validate the format of record
check if the content in record match the schema, if there is missing setion or not...

in each the record

In [96]:
# Validate whether one record matches key schema expectations.
# This is written to support future user-provided records too.

def _is_missing(value):
    if value is None:
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False


def _collect_required_unit_attributes(attr_mapping_df):
    """Return attribute names whose unit is defined in mapping and should be present in record."""
    if not isinstance(attr_mapping_df, pd.DataFrame):
        return set()

    required = set()
    required_cols = {"attribute_name", "unit"}
    if not required_cols.issubset(attr_mapping_df.columns):
        return required

    for _, rec in attr_mapping_df.iterrows():
        name = rec.get("attribute_name")
        unit = rec.get("unit")
        if _is_missing(name) or _is_missing(unit):
            continue
        unit_text = str(unit).strip()
        if unit_text != "-":
            required.add(str(name).strip())
    return required


def validate_record_against_schema(record_obj, schema_obj=None, attr_mapping_df=None):
    """Validate record with practical rules derived from current schema and mapping.

    Rules checked:
    1) Required top-level sections exist.
    2) Required nested info exists (technology/scope/balancing minimum keys).
    3) attributes items have required keys (attribute_name, value).
    4) Unit is present when attribute unit is required by mapping.
    5) balancing items have carrier_name and share.
    """
    errors = []
    warnings = []

    if not isinstance(record_obj, dict):
        return {
            "valid": False,
            "errors": ["record must be a dictionary"],
            "warnings": [],
        }

    # Rule 1: required top-level keys
    required_top = [
        "technology_name",
        "technology",
        "scope",
        "attributes",
        "balancing",
    ]
    for key in required_top:
        if key not in record_obj:
            errors.append(f"missing top-level key: {key}")

    # Rule 2: required nested keys
    tech_obj = record_obj.get("technology", {})
    if isinstance(tech_obj, dict):
        for key in ["technology_type", "process_name"]:
            if _is_missing(tech_obj.get(key)):
                errors.append(f"missing technology.{key}")
    else:
        errors.append("technology must be a dictionary")

    scope_obj = record_obj.get("scope", {})
    if isinstance(scope_obj, dict):
        if _is_missing(scope_obj.get("geographic_scope_description")):
            warnings.append("scope.geographic_scope_description is empty")
    else:
        errors.append("scope must be a dictionary")

    bal_obj = record_obj.get("balancing", {})
    if isinstance(bal_obj, dict):
        if "inputs" not in bal_obj and "outputs" not in bal_obj:
            errors.append("balancing must have inputs and/or outputs")
        for side in ["inputs", "outputs"]:
            side_items = bal_obj.get(side, [])
            if side_items is None:
                side_items = []
            if not isinstance(side_items, list):
                errors.append(f"balancing.{side} must be a list")
                continue
            for i, item in enumerate(side_items):
                if not isinstance(item, dict):
                    errors.append(f"balancing.{side}[{i}] must be a dict")
                    continue
                if _is_missing(item.get("carrier_name")):
                    errors.append(f"balancing.{side}[{i}].carrier_name is missing")
                if _is_missing(item.get("share")):
                    errors.append(f"balancing.{side}[{i}].share is missing")
    else:
        errors.append("balancing must be a dictionary")

    # Rule 3 + 4: values structure + required unit when mapping says so
    required_unit_attrs = _collect_required_unit_attributes(attr_mapping_df)
    values_obj = record_obj.get("attributes", [])
    if not isinstance(values_obj, list):
        errors.append("attributes must be a list")
    else:
        seen_attr_names = set()
        for i, item in enumerate(values_obj):
            if not isinstance(item, dict):
                errors.append(f"attributes[{i}] must be a dict")
                continue

            attr_name = item.get("attribute_name")
            attr_value = item.get("value")
            attr_unit = item.get("unit")

            if _is_missing(attr_name):
                errors.append(f"attributes[{i}].attribute_name is missing")
                continue

            attr_name = str(attr_name).strip()
            seen_attr_names.add(attr_name)

            if attr_value is None:
                errors.append(f"attributes[{i}].value is missing for {attr_name}")

            if attr_name in required_unit_attrs and _is_missing(attr_unit):
                errors.append(f"attributes[{i}].unit is required for {attr_name}")

        if len(seen_attr_names) == 0:
            warnings.append("attribute list is empty")

    # Optional Rule 5: alignment with schema keys if schema is available
    if isinstance(schema_obj, dict):
        if "unmapped_record" not in schema_obj:
            warnings.append("schema does not contain unmapped_record section")

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
    }

In [97]:
## load the data from ConvTech sheet for testing
df_conv = refuel_data['ConvTech'].copy()
df_conv.columns = df_conv.loc[2]
df_conv = df_conv.loc[3:].reset_index(drop=True)

row = df_conv.loc[9]

## run record mapping for the testing row
record = map_technology(row)

## run record validation
validation_result = validate_record_against_schema(
    record_obj=record,
    schema_obj=schema_motel,
    attr_mapping_df=df_attr_mapping,
)

## show validation results
print("is_valid:", validation_result["valid"] )
print("error_count:", len(validation_result["errors"]))
print("warning_count:", len(validation_result["warnings"]))

if validation_result["errors"]:
    print("\nErrors:")
    for msg in validation_result["errors"][:20]:
        print("-", msg)

if validation_result["warnings"]:
    print("\nWarnings:")
    for msg in validation_result["warnings"][:20]:
        print("-", msg)

is_valid: True
error_count: 0
warning_count: 0


In [98]:
record

{'technology_name': 'CarbonCapture Direct 2050',
 'technology': {'technology_description': 'Direct air capture using adsorption technology (2050).',
  'technology_type': 'Conversion',
  'technology_assumption': '2050 reference technology',
  'process_name': 'DAC',
  'process_type': 'CO2 capture'},
 'scope': {'geographic_scope_description': 'CH',
  'temporal_scope_description': '2050',
  'capacity_scope_description': 0,
  'scope_assumption': 'Emerging'},
 'sources': [{'source_description': 'parameters_lca',
   'source_type': 'database',
   'assessment_method': 'modeled'}],
 'attributes': [{'attribute_name': 'temp_operating',
   'attribute_description': 'Typical operating temperature of the process.',
   'unit': '°C',
   'value': 0},
  {'attribute_name': 'capacity_min',
   'attribute_description': 'Minimum viable installation capacity.',
   'unit': 'kg/h',
   'value': 0},
  {'attribute_name': 'lifetime_economic',
   'attribute_description': 'Economic lifetime of the asset.',
   'unit': '

In [99]:
row

2
Review done                             True
1st Review responsible             Christian
2nd Review responsible                   NaN
Review approved (Arash)                False
Review approved (Robin)                False
                                     ...    
Lifetime / Amortisation VSE2025          NaN
Difference                               NaN
Efficiency                               NaN
Difference                               NaN
NaN                                      NaN
Name: 9, Length: 80, dtype: object

## 3.3 Run record creating and validation for all reFuel.ch data

In [107]:
## run maping

records = []
sheet_name = 'ConvTech'

# read row by row (per technology)
df_conv = refuel_data[sheet_name]
df_conv.columns = df_conv.loc[2]
df_conv = df_conv.loc[3:].reset_index(drop=True)


for indx in df_conv.index:
    row = df_conv.loc[indx]    

    record = map_technology(row)
    print(f"Record for technology '{record['technology_name']}':")


    validation_result = validate_record_against_schema(
        record_obj=record,
        schema_obj=schema_motel,
        attr_mapping_df=df_attr_mapping,
    )

    # add validation result to the record
    if validation_result["valid"]:
        record['validation'] = "valid"
    else:
        print(f'index = {indx}')
        record['validation'] = "invalid"
        record['validation_errors'] = validation_result["errors"]
        record['validation_warnings'] = validation_result["warnings"]

        if validation_result["errors"]:
            print("\nErrors:")
            for msg in validation_result["errors"][:20]:
                print("-", msg)

        if validation_result["warnings"]:
            print("\nWarnings:")
            for msg in validation_result["warnings"][:20]:
                print("-", msg)

    records.append(record)

Record for technology 'Amine Washing CO2 2030':
Record for technology 'Amine Washing CO2 2040':
Record for technology 'Amine Washing CO2 2050':
Record for technology 'Ammonia CCGT 2050':
Record for technology 'Ammonia HaberBosch 2050':
Record for technology 'Ammonia OCGT 2050':
Record for technology 'BioWet Pyrolysis CBio':
Record for technology 'CarbonCapture Direct 2030':
Record for technology 'CarbonCapture Direct 2040':
Record for technology 'CarbonCapture Direct 2050':
Record for technology 'CarbonCapture PressureSwing 2030':
Record for technology 'CarbonCapture PressureSwing 2040':
Record for technology 'CarbonCapture PressureSwing 2050':
Record for technology 'CH4 CCGT 2050':
Record for technology 'CH4 CCGT CCS 2050':
index = 14

Errors:
- missing technology.technology_type

Warnings:
- scope.geographic_scope_description is empty
Record for technology 'CH4 CHP 2050':
Record for technology 'CH4 Digestion 2030':
Record for technology 'CH4 Digestion 2040':
Record for technology 'CH

In [108]:
df_conv.loc[14]

2
Review done                        False
1st Review responsible             Arash
2nd Review responsible             Robin
Review approved (Arash)             True
Review approved (Robin)            False
                                   ...  
Lifetime / Amortisation VSE2025      NaN
Difference                           NaN
Efficiency                           NaN
Difference                           NaN
NaN                                  NaN
Name: 14, Length: 80, dtype: object

In [101]:
records


[{'technology_name': 'Amine Washing CO2 2030',
  'technology': {'technology_description': 'CO2 post combustion capture via Amine washing. Temperature swing process. 2030 reference based on an approx. 150 ktCO2/a plant size. ',
   'technology_type': 'Conversion',
   'technology_category': 'Point source capture',
   'technology_assumption': '2030 reference technology',
   'process_name': 'Amine Absorber',
   'process_type': 'CO2 capture',
   'process_category': 'CO2 capture',
   'process_assumption': 'Amine washing'},
  'scope': {'geographic_scope_description': 'CH',
   'temporal_scope_description': '2030',
   'capacity_scope_description': '20000 kg/h',
   'system_boundary_description': 'Plant ready to operate',
   'scope_assumption': 'Mature'},
  'sources': [{'source_description': 'AECOM_2022',
    'source_type': 'report',
    'assessment_method': 'literature'},
   {'source_description': 'parameters_lca',
    'source_type': 'database',
    'assessment_method': 'modeled'}],
  'attributes